In [12]:

import requests
import pandas as pd
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ==========================================
# STEP 1: WEB SCRAPING
# Maximum of 200 unique books
# ==========================================

BASE_URL = "https://books.toscrape.com/"
MAX_BOOKS = 200

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# Create a session for webpage requests
session = requests.Session()
session.headers.update(HEADERS)


def get_soup(url):
    """Download a webpage and convert it to BeautifulSoup."""
    response = session.get(url, timeout=30)
    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")


def get_rating(book):
    """Convert a star rating into a number from 1 to 5."""
    rating_element = book.select_one("p.star-rating")

    if rating_element is None:
        return None

    rating_classes = rating_element.get("class", [])

    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    for word, number in rating_map.items():
        if word in rating_classes:
            return number

    return None


def scrape_books():
    """Collect a maximum of 200 unique books."""
    books = []
    seen_urls = set()

    # Get the homepage
    soup = get_soup(BASE_URL)

    # Collect category links in displayed order
    category_links = soup.select(
        "div.side_categories ul li ul li a"
    )

    if not category_links:
        raise RuntimeError("No category links found.")

    for category_link in category_links:

        if len(books) >= MAX_BOOKS:
            break

        category = category_link.get_text(strip=True)

        category_url = urljoin(
            BASE_URL,
            category_link.get("href", "")
        )

        next_page = category_url

        while next_page and len(books) < MAX_BOOKS:

            print(
                f"Scraping {category}: "
                f"{len(books)}/{MAX_BOOKS} books collected"
            )

            page = get_soup(next_page)

            book_cards = page.select("article.product_pod")

            for card in book_cards:

                if len(books) >= MAX_BOOKS:
                    break

                title_element = card.select_one("h3 a")
                price_element = card.select_one(
                    "p.price_color"
                )

                if title_element is None or price_element is None:
                    continue

                title = (
                    title_element.get("title")
                    or title_element.get_text(strip=True)
                )

                book_url = urljoin(
                    next_page,
                    title_element.get("href", "")
                )

                # Avoid collecting the same book twice
                if book_url in seen_urls:
                    continue

                seen_urls.add(book_url)

                price_text = price_element.get_text(strip=True)

                # Website prices are displayed in GBP (£)
                try:
                    price = float(
                        price_text.replace("£", "").strip()
                    )
                except ValueError:
                    price = None

                rating = get_rating(card)

                availability_element = card.select_one(
                    "p.instock.availability"
                )

                availability = (
                    availability_element.get_text(
                        " ", strip=True
                    )
                    if availability_element
                    else "Unknown"
                )

                books.append({
                    "title": title,
                    "price_gbp": price,
                    "rating": rating,
                    "category": category,
                    "availability": availability,
                    "book_url": book_url
                })

            # Find the next page in the current category
            next_link = page.select_one("li.next a")

            if next_link and next_link.get("href"):
                next_page = urljoin(
                    next_page,
                    next_link["href"]
                )
            else:
                next_page = None

            time.sleep(0.15)

    # Final safety check: maximum of 200 records
    books = books[:MAX_BOOKS]

    return books


def main():
    print("BOOK MARKET ANALYTICS")
    print("Starting web scraping...")
    print(f"Maximum books: {MAX_BOOKS}")
    print("-" * 40)

    try:
        books = scrape_books()

        if not books:
            print("No books were collected.")
            return

        # Convert the collected books into a DataFrame
        dataframe = pd.DataFrame(books)

        # Save the dataset
        dataframe.to_csv(
            "books_raw.csv",
            index=False,
            encoding="utf-8-sig"
        )

        print("\nSCRAPING COMPLETE")
        print(f"Unique books collected: {len(dataframe)}")
        print("File created: books_raw.csv")

        print("\nFirst five books:")
        print(dataframe.head().to_string(index=False))

        print("\nDataset information:")
        print(dataframe.info())

    except requests.RequestException as error:
        print(f"Website connection error: {error}")

    except Exception as error:
        print(f"An error occurred: {error}")


if __name__ == "__main__":
    main()


BOOK MARKET ANALYTICS
Starting web scraping...
Maximum books: 200
----------------------------------------
Scraping Travel: 0/200 books collected
Scraping Mystery: 11/200 books collected
Scraping Mystery: 31/200 books collected
Scraping Historical Fiction: 43/200 books collected
Scraping Historical Fiction: 63/200 books collected
Scraping Sequential Art: 69/200 books collected
Scraping Sequential Art: 89/200 books collected
Scraping Sequential Art: 109/200 books collected
Scraping Sequential Art: 129/200 books collected
Scraping Classics: 144/200 books collected
Scraping Philosophy: 163/200 books collected
Scraping Romance: 174/200 books collected
Scraping Romance: 194/200 books collected

SCRAPING COMPLETE
Unique books collected: 200
File created: books_raw.csv

First five books:
                                                              title price_gbp  rating category availability                                                                                                     

In [13]:

import pandas as pd
from pathlib import Path

# ==========================================
# STEP 2: DATA CLEANING
# ==========================================

INPUT_FILE = Path("books_raw.csv")
OUTPUT_FILE = Path("books_cleaned.csv")


def clean_books():
    """Clean the raw book dataset and save the results."""

    # 1. Check whether the raw dataset exists
    if not INPUT_FILE.exists():
        print("ERROR: books_raw.csv was not found.")
        print("Run step1_web_scraping.py first.")
        return

    # 2. Load the raw dataset
    df = pd.read_csv(INPUT_FILE)

    print("=" * 50)
    print("STEP 2: DATA CLEANING")
    print("=" * 50)

    print(f"\nOriginal number of rows: {len(df)}")

    # 3. Display missing values before cleaning
    print("\nMissing values before cleaning:")
    print(df.isnull().sum())

    # 4. Remove spaces around text
    text_columns = [
        "title",
        "category",
        "availability",
        "book_url"
    ]

    for column in text_columns:
        if column in df.columns:
            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
            )

            # Treat blank strings as missing
            df[column] = df[column].replace(
                "", pd.NA
            )

    # 5. Convert price and rating to numeric values
    for column in ["price_gbp", "rating"]:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

    # 6. Remove duplicate books
    if "book_url" in df.columns:
        valid_urls = df["book_url"].notna()

        duplicates = df.loc[
            valid_urls
        ].duplicated(
            subset=["book_url"],
            keep="first"
        )

        duplicate_indexes = duplicates[
            duplicates
        ].index

        df = df.drop(index=duplicate_indexes)

    else:
        df = df.drop_duplicates(
            subset=["title", "category"],
            keep="first"
        )

    print(
        "\nRows after duplicate removal: "
        f"{len(df)}"
    )

    # 7. Remove records without a title or price
    required_columns = [
        column
        for column in ["title", "price_gbp"]
        if column in df.columns
    ]

    df = df.dropna(
        subset=required_columns
    )

    # 8. Remove invalid prices
    df = df[df["price_gbp"] >= 0]

    # 9. Keep ratings from 1 to 5, or missing ratings
    if "rating" in df.columns:
        valid_rating = (
            df["rating"].between(1, 5)
            | df["rating"].isna()
        )

        df = df[valid_rating]

    # 10. Standardise category and availability
    for column in ["category", "availability"]:
        if column in df.columns:
            df[column] = (
                df[column]
                .fillna("Unknown")
                .replace("", "Unknown")
            )

    # 11. Make ratings whole numbers where present
    if "rating" in df.columns:
        df["rating"] = df["rating"].astype("Int64")

    # 12. Keep the dataset within the 200-book limit
    df = df.head(200).reset_index(drop=True)

    # 13. Save the cleaned dataset
    df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    # 14. Display cleaning results
    print("\nCLEANING COMPLETE")
    print("-" * 50)
    print(f"Final number of rows: {len(df)}")
    print(f"Final number of columns: {len(df.columns)}")

    print("\nMissing values after cleaning:")
    print(df.isnull().sum())

    print("\nCleaned dataset preview:")
    print(df.head(10).to_string(index=False))

    print(f"\nCleaned dataset saved as: {OUTPUT_FILE}")

    print("\nColumn data types:")
    print(df.dtypes)

    return df


if __name__ == "__main__":
    clean_books()


STEP 2: DATA CLEANING

Original number of rows: 200

Missing values before cleaning:
title             0
price_gbp       200
rating            0
category          0
availability      0
book_url          0
dtype: int64

Rows after duplicate removal: 200

CLEANING COMPLETE
--------------------------------------------------
Final number of rows: 0
Final number of columns: 6

Missing values after cleaning:
title           0
price_gbp       0
rating          0
category        0
availability    0
book_url        0
dtype: int64

Cleaned dataset preview:
Empty DataFrame
Columns: [title, price_gbp, rating, category, availability, book_url]
Index: []

Cleaned dataset saved as: books_cleaned.csv

Column data types:
title            string
price_gbp       float64
rating            Int64
category         string
availability     string
book_url         string
dtype: object


In [14]:

import pandas as pd
import matplotlib

# Allows graphs to be saved without opening separate windows
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


# ==========================================
# STEP 3: EXPLORATORY DATA ANALYSIS
# ==========================================

INPUT_FILE = Path("books_cleaned.csv")
CHART_FOLDER = Path("charts")

CHART_FOLDER.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid")


def analyse_books():

    # 1. Load the cleaned dataset
    if not INPUT_FILE.exists():
        print("ERROR: books_cleaned.csv was not found.")
        print("Run Step 2 first.")
        return

    df = pd.read_csv(INPUT_FILE)

    if df.empty:
        print("ERROR: The cleaned dataset is empty.")
        return

    # 2. Check that the required columns exist
    required_columns = [
        "title",
        "price_gbp",
        "rating",
        "category",
        "availability"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        print("Missing columns:", missing_columns)
        return

    # 3. Convert numeric columns
    df["price_gbp"] = pd.to_numeric(
        df["price_gbp"],
        errors="coerce"
    )

    df["rating"] = pd.to_numeric(
        df["rating"],
        errors="coerce"
    )

    df = df.dropna(subset=["price_gbp"])

    if df.empty:
        print("No valid book prices are available.")
        return

    print("=" * 55)
    print("STEP 3: EXPLORATORY DATA ANALYSIS")
    print("=" * 55)

    # ==========================================
    # PART A: SUMMARY STATISTICS
    # ==========================================

    print("\n1. DATASET OVERVIEW")
    print("-" * 40)

    print(f"Total books analysed: {len(df)}")
    print(f"Number of categories: {df['category'].nunique()}")

    print("\n2. PRICE STATISTICS")
    print("-" * 40)

    print(f"Average price: £{df['price_gbp'].mean():.2f}")
    print(f"Median price: £{df['price_gbp'].median():.2f}")
    print(f"Lowest price: £{df['price_gbp'].min():.2f}")
    print(f"Highest price: £{df['price_gbp'].max():.2f}")
    print(f"Price standard deviation: £{df['price_gbp'].std():.2f}")

    print("\n3. RATING STATISTICS")
    print("-" * 40)

    print(f"Books with ratings: {df['rating'].notna().sum()}")

    if df["rating"].notna().any():
        print(f"Average rating: {df['rating'].mean():.2f} / 5")
    else:
        print("Average rating: unavailable")

    print("\n4. NUMBER OF BOOKS PER CATEGORY")
    print("-" * 40)

    print(
        df["category"]
        .value_counts()
        .to_string()
    )

    print("\n5. STOCK STATUS")
    print("-" * 40)

    print(
        df["availability"]
        .value_counts()
        .to_string()
    )

    # Save numerical summary
    summary = df[["price_gbp", "rating"]].describe()

    summary.to_csv("eda_summary.csv")

    # ==========================================
    # GRAPH 1: PRICE DISTRIBUTION
    # ==========================================

    plt.figure(figsize=(9, 5))

    sns.histplot(
        data=df,
        x="price_gbp",
        bins=20,
        kde=True
    )

    plt.title("Distribution of Book Prices")
    plt.xlabel("Book Price (£)")
    plt.ylabel("Number of Books")
    plt.tight_layout()

    plt.savefig(
        CHART_FOLDER / "graph1_price_distribution.png",
        dpi=150
    )

    plt.close()

    # ==========================================
    # GRAPH 2: RATING DISTRIBUTION
    # ==========================================

    rating_data = df.dropna(subset=["rating"]).copy()

    if not rating_data.empty:

        rating_data = rating_data[
            rating_data["rating"].between(1, 5)
        ]

        rating_counts = (
            rating_data["rating"]
            .astype(int)
            .value_counts()
            .reindex([1, 2, 3, 4, 5], fill_value=0)
        )

        plt.figure(figsize=(8, 5))

        sns.barplot(
            x=rating_counts.index,
            y=rating_counts.values
        )

        plt.title("Distribution of Book Ratings")
        plt.xlabel("Star Rating")
        plt.ylabel("Number of Books")
        plt.xticks([0, 1, 2, 3, 4], [1, 2, 3, 4, 5])
        plt.tight_layout()

        plt.savefig(
            CHART_FOLDER / "graph2_rating_distribution.png",
            dpi=150
        )

        plt.close()

    else:
        print("\nGraph 2 skipped: no ratings available.")

    # ==========================================
    # GRAPH 3: AVERAGE PRICE BY CATEGORY
    # ==========================================

    category_prices = (
        df.groupby("category", dropna=False)
        .agg(
            average_price=("price_gbp", "mean"),
            number_of_books=("title", "count")
        )
        .sort_values("average_price", ascending=False)
    )

    # Limit the graph to the 10 categories
    # with the most books, for readability.
    top_categories = (
        df["category"]
        .value_counts()
        .head(10)
        .index
    )

    plot_data = category_prices.loc[
        category_prices.index.isin(top_categories)
    ].sort_values("average_price", ascending=True)

    plt.figure(figsize=(10, 6))

    sns.barplot(
        data=plot_data.reset_index(),
        x="average_price",
        y="category"
    )

    plt.title("Average Book Price by Category")
    plt.xlabel("Average Price (£)")
    plt.ylabel("Book Category")
    plt.tight_layout()

    plt.savefig(
        CHART_FOLDER / "graph3_category_prices.png",
        dpi=150
    )

    plt.close()

    print("\nAVERAGE PRICE BY CATEGORY")
    print("-" * 40)

    print(
        category_prices
        .round(2)
        .to_string()
    )

    # Save category statistics
    category_prices.to_csv("category_summary.csv")

    # ==========================================
    # GRAPH 4: STOCK AVAILABILITY
    # ==========================================

    stock_counts = (
        df["availability"]
        .fillna("Unknown")
        .value_counts()
    )

    plt.figure(figsize=(8, 5))

    sns.barplot(
        x=stock_counts.index,
        y=stock_counts.values
    )

    plt.title("Book Stock Availability")
    plt.xlabel("Availability Status")
    plt.ylabel("Number of Books")
    plt.xticks(rotation=15)
    plt.tight_layout()

    plt.savefig(
        CHART_FOLDER / "graph4_stock_availability.png",
        dpi=150
    )

    plt.close()

    # ==========================================
    # COMPLETE
    # ==========================================

    print("\nSTEP 3 COMPLETED SUCCESSFULLY")
    print("-" * 40)
    print(f"Graphs saved in: {CHART_FOLDER.resolve()}")
    print("Summary saved as: eda_summary.csv")
    print("Category data saved as: category_summary.csv")

    print("\nGenerated chart files:")

    for chart in sorted(CHART_FOLDER.glob("graph*.png")):
        print(f"- {chart.name}")


if __name__ == "__main__":
    analyse_books()


ERROR: The cleaned dataset is empty.


In [15]:

import pandas as pd
import numpy as np
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ==========================================
# STEP 4: LEARNING
# ==========================================

INPUT_FILE = Path("books_cleaned.csv")
OUTPUT_PREDICTIONS = Path("test_predictions.csv")
OUTPUT_RESULTS = Path("model_results.txt")
OUTPUT_CHART = Path("charts/model_predictions.png")

OUTPUT_CHART.parent.mkdir(
    parents=True,
    exist_ok=True
)


def train_price_model():

    # 1. Load the cleaned dataset
    if not INPUT_FILE.exists():
        print("ERROR: books_cleaned.csv was not found.")
        print("Run Steps 1 and 2 first.")
        return

    df = pd.read_csv(INPUT_FILE)

    print("=" * 55)
    print("STEP 4: MACHINE LEARNING")
    print("=" * 55)

    print(f"\nBooks loaded: {len(df)}")

    # 2. Check required columns
    required_columns = [
        "price_gbp",
        "rating",
        "category",
        "stock_quantity"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    # The scraper in Step 1 does not collect stock_quantity.
    # If absent, create it as missing; the model pipeline
    # will handle the missing values.
    if "stock_quantity" in missing_columns:
        df["stock_quantity"] = np.nan
        missing_columns.remove("stock_quantity")

    if missing_columns:
        print(
            "ERROR: Missing required columns:",
            missing_columns
        )
        return

    # 3. Convert numerical fields
    df["price_gbp"] = pd.to_numeric(
        df["price_gbp"],
        errors="coerce"
    )

    df["rating"] = pd.to_numeric(
        df["rating"],
        errors="coerce"
    )

    df["stock_quantity"] = pd.to_numeric(
        df["stock_quantity"],
        errors="coerce"
    )

    # 4. Remove rows without a valid target price
    df = df.dropna(subset=["price_gbp"])
    df = df[df["price_gbp"] >= 0]

    # Keep only plausible ratings; missing ratings are allowed.
    df.loc[
        ~df["rating"].between(1, 5),
        "rating"
    ] = np.nan

    df["category"] = (
        df["category"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
    )

    if len(df) < 10:
        print(
            "ERROR: At least 10 books with valid prices "
            "are required for this exercise."
        )
        return

    # 5. Select features (X) and target (y)
    feature_columns = [
        "rating",
        "category",
        "stock_quantity"
    ]

    X = df[feature_columns]
    y = df["price_gbp"]

    print(f"Usable records: {len(df)}")
    print("\nInput features:")
    print(feature_columns)
    print("Target variable: price_gbp")

    # 6. Build preprocessing pipelines
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore")
            )
        ]
    )

    preprocessing = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                ["rating", "stock_quantity"]
            ),
            (
                "categorical",
                categorical_pipeline,
                ["category"]
            )
        ]
    )

    # 7. Create the Linear Regression model
    model = Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("regression", LinearRegression())
        ]
    )

    # 8. Split into training and testing datasets
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    print(f"\nTraining records: {len(X_train)}")
    print(f"Testing records: {len(X_test)}")

    # 9. Train the model
    model.fit(X_train, y_train)

    print("\nModel training completed.")

    # 10. Predict prices for the test set
    predictions = model.predict(X_test)

    # 11. Evaluate the model
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    print("\nMODEL EVALUATION")
    print("-" * 40)
    print(f"Mean Absolute Error (MAE): £{mae:.2f}")
    print(f"Root Mean Squared Error (RMSE): £{rmse:.2f}")
    print(f"R-squared (R²): {r2:.4f}")

    # 12. Save predictions
    results_df = pd.DataFrame({
        "actual_price_gbp": y_test.to_numpy(),
        "predicted_price_gbp": predictions,
        "absolute_error_gbp": np.abs(
            y_test.to_numpy() - predictions
        )
    })

    results_df.to_csv(
        OUTPUT_PREDICTIONS,
        index=False
    )

    # 13. Save evaluation metrics
    with open(
        OUTPUT_RESULTS,
        "w",
        encoding="utf-8"
    ) as file:

        file.write("BOOK MARKET ANALYTICS\n")
        file.write("LINEAR REGRESSION RESULTS\n")
        file.write("=" * 40 + "\n")

        file.write(f"Records used: {len(df)}\n")
        file.write(f"Training records: {len(X_train)}\n")
        file.write(f"Testing records: {len(X_test)}\n")

        file.write(f"MAE: £{mae:.2f}\n")
        file.write(f"RMSE: £{rmse:.2f}\n")
        file.write(f"R-squared: {r2:.4f}\n")

        file.write("\nTarget: book price in GBP\n")
        file.write(
            "Features: rating, category, stock quantity\n"
        )

    # 14. Create actual-versus-predicted graph
    plt.figure(figsize=(8, 6))

    plt.scatter(
        y_test,
        predictions,
        alpha=0.8
    )

    # Reference line: perfect predictions
    minimum = min(
        float(y_test.min()),
        float(np.min(predictions))
    )

    maximum = max(
        float(y_test.max()),
        float(np.max(predictions))
    )

    plt.plot(
        [minimum, maximum],
        [minimum, maximum],
        linestyle="--"
    )

    plt.title("Actual vs Predicted Book Prices")
    plt.xlabel("Actual Price (£)")
    plt.ylabel("Predicted Price (£)")
    plt.tight_layout()

    plt.savefig(
        OUTPUT_CHART,
        dpi=150
    )

    plt.close()

    # 15. Display a sample of predictions
    print("\nSAMPLE PREDICTIONS")
    print("-" * 40)

    print(
        results_df.head(10).round(2).to_string(index=False)
    )

    print("\nSTEP 4 COMPLETED")
    print(f"Predictions saved to: {OUTPUT_PREDICTIONS}")
    print(f"Metrics saved to: {OUTPUT_RESULTS}")
    print(f"Chart saved to: {OUTPUT_CHART}")

    print(
        "\nNote: these metrics measure performance on the "
        "held-out test data. They may vary if the dataset changes."
    )


if __name__ == "__main__":
    train_price_model()


STEP 4: MACHINE LEARNING

Books loaded: 0
ERROR: At least 10 books with valid prices are required for this exercise.
